# 第 1 節｜OpenCV 影像處理與車牌 OCR

**第 1 節｜約 20 分鐘核心練習＋車牌 OCR 選做。** 本節從 OpenCV 基本操作開始，再用虛構車牌完成「找範圍 → 擷取 → OCR」。

將 ZIP 解壓縮到 Colab 的 `/content/`，保留 `04_素材` 資料夾；若從 VS Code 連到 Colab，請確認目前工作目錄包含此資料夾。

## 1. 套件安裝與模組匯入

**先執行套件安裝 Cell，再執行下一格 `import`。** Colab／VS Code 連線到 Colab 的 Python 環境都使用相同安裝方式。

In [ ]:
# Install the system OCR engine in Colab.
# 在 Colab 安裝辨識車牌文字的 OCR 引擎。
!apt-get -qq update > /dev/null && apt-get -qq install -y tesseract-ocr > /dev/null

# Install the Python interface for OCR.
# 安裝 Python 使用 Tesseract 的介面。
%pip -q install pytesseract

In [ ]:
# Import helpers for image file paths.
# 匯入圖片檔案路徑處理工具。
from pathlib import Path

# Import OpenCV, NumPy, and the OCR interface.
# 匯入 OpenCV、NumPy 與 OCR 介面。
import cv2, numpy as np, pytesseract

# Import Matplotlib for plots.
# 匯入 Matplotlib 繪製圖表。
import matplotlib.pyplot as plt

# Import Pillow for image display.
# 匯入 Pillow 方便顯示圖片。
from PIL import Image

# Import notebook display support.
# 匯入 Notebook 圖片顯示工具。
from IPython.display import display

## 2. 看懂影像、尺寸與 BGR/RGB

**OpenCV 預設 BGR；Matplotlib 顯示照片時使用 RGB。** 修改 `IMAGE_PATH` 可以改成同資料夾內的 `cat.jpg` 或 `coffee.jpg`。

In [ ]:
# Choose a photograph from the slide materials.
# 選擇投影片內的實際照片。
IMAGE_PATH = Path("04_素材/motorcycle.jpg")

# Read the image using OpenCV's BGR channel order.
# 使用 OpenCV 載入照片，通道順序為 BGR。
img = cv2.imread(str(IMAGE_PATH))

# Show the original photograph with correct RGB colors.
# 將 BGR 轉成 RGB，顯示正確顏色。
display(Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)))

In [ ]:
# Print the image shape: height, width, and channels.
# 檢查影像的高度、寬度與色彩通道數。
print("影像尺寸 (高、寬、通道)：", img.shape)

# Display the BGR pixel in the image center.
# 觀察影像中央像素的 BGR 數值。
print("中央像素 BGR：", img[img.shape[0] // 2, img.shape[1] // 2])

## 3. 縮放、灰階、模糊與邊緣

觀察照片被縮小後，以及透過灰階、Gaussian Blur 與 Canny 處理後有什麼差異。

In [ ]:
# Resize the image to half its width and height.
# 將影像的寬度與高度各縮小為一半。
small = cv2.resize(img, None, fx=0.5, fy=0.5)

# Convert the image to grayscale.
# 將彩色影像轉換為灰階影像。
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# Display the resized color image.
# 顯示縮小後的彩色影像。
display(Image.fromarray(cv2.cvtColor(small, cv2.COLOR_BGR2RGB)))

# Display the grayscale version for comparison.
# 顯示灰階版本，方便比較。
display(Image.fromarray(gray))

In [ ]:
# Reduce noise with Gaussian blur.
# 使用 Gaussian Blur 平滑影像，減少雜訊。
blur = cv2.GaussianBlur(gray, (7, 7), 0)

# Find strong edges using Canny.
# 使用 Canny 找出亮度變化明顯的邊緣。
edges = cv2.Canny(blur, 80, 160)

# Show blurred pixels above the detected edges.
# 先顯示平滑影像，再顯示邊緣影像。
display(Image.fromarray(blur))

# Observe which details remain as edges.
# 觀察哪些細節仍保留成邊緣線條。
display(Image.fromarray(edges))

## 4. 亮度直方圖與二值化

直方圖表示「每種亮度有多少像素」；二值化則依門檻把像素分成黑、白兩種。

In [ ]:
# Draw the histogram of grayscale pixel intensities.
# 統計灰階像素從暗到亮的分布。
plt.hist(gray.ravel(), bins=64, range=(0, 256))

# Show the distribution as a chart.
# 顯示灰階亮度直方圖。
plt.show()

# Convert grayscale values to a black-and-white image.
# 以門檻 130 將灰階影像轉成黑白兩色。
_, binary = cv2.threshold(gray, 130, 255, cv2.THRESH_BINARY)

# Display the threshold result.
# 顯示二值化後的影像。
display(Image.fromarray(binary))

## 5. 傳統視覺：用顏色找紅色區域

這是**顏色規則**，不能等同於理解「機車」；紅色衣服與招牌也可能被選中。

In [ ]:
# Convert BGR pixels into the HSV color space.
# 將 BGR 影像轉換為 HSV，方便設定顏色範圍。
hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

# Detect the low-hue red range.
# 偵測 HSV 色相低值範圍的紅色。
red1 = cv2.inRange(hsv, (0, 90, 70), (12, 255, 255))

# Detect the high-hue red range and merge both masks.
# 加入另一段紅色色相範圍，合併遮罩。
red_mask = red1 | cv2.inRange(hsv, (170, 90, 70), (179, 255, 255))

In [ ]:
# Keep only the regions marked red.
# 僅保留紅色遮罩選到的影像區域。
red_only = cv2.bitwise_and(img, img, mask=red_mask)

# Display the red regions instead of classifying the whole object.
# 顯示紅色區域，觀察顏色規則的限制。
display(Image.fromarray(cv2.cvtColor(red_only, cv2.COLOR_BGR2RGB)))

## 6. 選做：車牌偵測、擷取與 OCR

資料夾中的 `plate_demo.png` 是**虛構車牌**，不是任何真實車主資料。OpenCV 找矩形候選區域；Tesseract 辨識裁切後的文字。真實車牌常受角度、反光、遮擋影響，此方法不保證能辨識。

In [ ]:
# Load the fictional license-plate example.
# 讀取教材提供的虛構車牌範例。
plate = cv2.imread("04_素材/plate_demo.png")

# Display the original image before plate extraction.
# 先觀察尚未擷取車牌的原始圖片。
display(Image.fromarray(cv2.cvtColor(plate, cv2.COLOR_BGR2RGB)))

In [ ]:
# Convert the photo to grayscale for contour detection.
# 轉成灰階，供輪廓偵測使用。
plate_gray = cv2.cvtColor(plate, cv2.COLOR_BGR2GRAY)

# Extract edges from high-contrast regions.
# 偵測高對比位置的邊緣。
plate_edges = cv2.Canny(plate_gray, 60, 160)

# Find candidate contours in the edge image.
# 在邊緣影像中找出候選輪廓。
contours, _ = cv2.findContours(plate_edges, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)

In [ ]:
# Prepare an empty list of possible license-plate rectangles.
# 建立清單，儲存可能的車牌矩形。
rectangles = []

# Examine each contour for a wide plate-like rectangle.
# 檢查輪廓是否符合寬扁的車牌形狀。
for contour in contours:
    # Measure the rectangular region around the contour.
    # 取得輪廓的矩形範圍。
    x, y, w, h = cv2.boundingRect(contour)
    # Keep reasonably large and wide candidates.
    # 保留夠大且符合車牌長寬比例的候選區域。
    if h > 0 and y > plate.shape[0] / 2 and 3 < w / h < 6 and w * h > 20000:
        rectangles.append((w * h, x, y, w, h))

In [ ]:
# Select the largest suitable plate candidate.
# 從候選矩形中挑出面積最大的車牌區域。
_, x, y, w, h = max(rectangles)

# Crop the plate region for OCR.
# 擷取車牌區域，準備辨識文字。
roi = plate_gray[y:y + h, x:x + w]

# Show the cropped plate.
# 顯示裁切後的車牌影像。
display(Image.fromarray(roi))

In [ ]:
# Recognize the single row of plate text.
# 將車牌視為單行文字，交給 Tesseract OCR。
number = pytesseract.image_to_string(roi, config="--psm 7 -c tessedit_char_whitelist=ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789-").strip()

# Print only the extracted text.
# 只顯示最重要的辨識結果。
print("辨識到的車牌：", number)

**課堂討論：**為什麼 OpenCV 找到車牌位置後，還需要 OCR？如果車牌傾斜或被遮住，可能在哪一步失敗？

**自己試試看：**可以把 `plate` 改成 `cv2.imread("04_素材/自己的照片.jpg")`；要使用已取得同意或公開授權的影像，並注意車牌資訊可能涉及隱私。